In [38]:
import os
import sys
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy import loadtxt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline, make_pipeline

In [23]:
dataset_druggable = pd.read_csv("input/combined_DepMap_21Q3_druggable.csv")

In [24]:
dataset_ccle = pd.read_csv("input/combined_DepMap_21Q3_CCLE_expression.csv")

In [28]:
dataset_ccle.iloc[1]

DepMap_ID                                      ACH-000242
TSPAN6 (7105)                                    7.465648
TNMD (64102)                                          0.0
DPM1 (8813)                                      6.435462
SCYL3 (57147)                                    2.414136
                                                  ...    
BRD-K99506538-001-03-8::2.5::MTS004                     0
BRD-K99616396-001-05-1::2.499991421::MTS004             0
BRD-K99879819-001-02-1::2.5187366::MTS004               0
BRD-K99919177-001-01-3::2.5::MTS004                     0
BRD-M63173034-001-03-6::2.64076472::MTS004              0
Name: 1, Length: 23864, dtype: object

In [29]:
dataset.iloc[1]

cell_line_name                                     786O
SHOC2                                          -0.21108
NDUFA12                                       -0.062542
SDAD1                                           -0.5537
FAM98A                                         -0.12355
                                                 ...   
BRD-K99506538-001-03-8::2.5::MTS004                   1
BRD-K99616396-001-05-1::2.499991421::MTS004           0
BRD-K99879819-001-02-1::2.5187366::MTS004             0
BRD-K99919177-001-01-3::2.5::MTS004                   0
BRD-M63173034-001-03-6::2.64076472::MTS004            1
Name: 1, Length: 22338, dtype: object

In [5]:
dataset = pd.read_csv("input/combined_DepMap_21Q3.csv")
num_gene = 17651
X = dataset.iloc[:, 1:num_gene+1]
drug_y_all = dataset.iloc[:, -4686:]
drug_list = drug_y_all.columns.tolist()

In [7]:
directory_path = "output/oversample"
os.makedirs(directory_path, exist_ok=True)

In [13]:
ROS = RandomOverSampler(random_state=72)
X_ovs, y_ovs = ROS.fit_resample(X, drug_y_all['BRD-A00100033-001-08-9::2.5::HTS'])

In [16]:
dataset.shape

(924, 22338)

In [15]:
X.shape

(924, 17651)

703

In [64]:
def xgbc(drug, oversample = True):
     y = drug_y_all[drug]

     

     
     # split X and y into training and testing sets
     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)


     if oversample:
          ros = RandomOverSampler(random_state=72)
          X_res, y_res = ros.fit_resample(X_train, y_train)
          oversample = 'oversample'
     else:
          X_res, y_res = X_train, y_train
          oversample = ''

     # declare parameters
     parameters = {
         "n_estimators": 150,
         "eta" : 0.1,
         "gamma" : 0.0,
         "max_depth" : 100,
         "eval_metric": "auc"
         }

     
     grid_search_parameters = {
         "n_estimators": [100, 150, 200],
         "eta" : [0.1],
         "gamma" : [0.0],
         "max_depth" : [20, 50, 100, 150],
         }

     # instantiate the classifier
     xgb0 = XGBClassifier(**parameters, use_label_encoder=False, n_jobs=10)

     # fit the classifier to the training data
     xgb0.fit(X_res, y_res)

     # save the trained model
     joblib.dump(xgb0, "output/"+oversample+"/XGBoost_%s.joblib" % drug)

    # make predictions on test data
     y_pred = xgb0.predict(X_test)

     print(drug,
         "XGBoost_model_parameters", xgb0, "\n",
         "confusion_matrix:", "\n", confusion_matrix(y_test, y_pred), "\n",
         file=open("output/"+oversample+"/confusion_matrix.txt", "a"))

     model_report = classification_report(y_test, y_pred, output_dict=True, labels=np.unique(y_pred))
     model_report = pd.DataFrame(model_report).transpose()
    
     if (model_report.index == "1").any() == True:
          r1 = pd.DataFrame(model_report.loc["1"]).transpose()
          r1.to_csv("output/"+oversample+"/classification_report_%s.csv" % drug)
     else:
          print(drug, "Nothing predicted as 1",
               file=open("output/"+oversample+"/classification_report_log.txt", "a"))

    # k-fold cross validation using multiple metric evaluation
     kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
     #cv_results = cross_validate(xgbc, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1'], n_jobs = 10)
     if oversample:
          imba_pipeline = Pipeline([('sampling', SMOTE(random_state=72)), 
                              ('classifier', XGBClassifier( n_jobs=10))])
          grid_search_parameters = {'classifier__' + key: grid_search_parameters[key] for key in grid_search_parameters}
     else:
          imba_pipeline = XGBClassifier( n_jobs=10)
     #cross_val_score(imba_pipeline, X_train, y_train, scoring='recall', cv=kf)
     grid_imba = GridSearchCV(imba_pipeline, param_grid=grid_search_parameters, cv=kfold, scoring='f1',
                        return_train_score=True)
     grid_imba.fit(X_train, y_train) 
     
     cv_results = cross_validate(imba_pipeline, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], n_jobs = 10)
     cv_results = pd.DataFrame(cv_results)
     cv_results.to_csv("output/"+oversample+"/cv_results_%s.csv" % drug)

     # feature importance with XGBoost
     fi = pd.DataFrame({'feature': list(X_train.columns),
               'importances': xgb0.feature_importances_ * 100}).\
                sort_values('importances', ascending = False)
     fi.to_csv("output/"+oversample+"/feature_importance_%s.csv" % drug)
     return grid_imba

In [65]:
starttime = time.time()
grid_search = xgbc('BRD-A00100033-001-08-9::2.5::HTS',  oversample = True
)
endtime = time.time()

/users/ysu13/miniforge3/envs/drug_sensitivity_ml/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [13:45:29] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1745056857893/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/users/ysu13/miniforge3/envs/drug_sensitivity_ml/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/users/ysu13/miniforge3/envs/drug_sen

In [57]:
test_pipeline = make_pipeline(SMOTE(random_state=72), 
                              XGBClassifier( n_jobs=10))

In [ ]:
pd.DataFrame(grid_search.cv_results_)[,5]

(12, 24)

In [50]:
starttime = time.time()
xgbc('BRD-A00100033-001-08-9::2.5::HTS'
)
endtime = time.time()

/users/ysu13/miniforge3/envs/drug_sensitivity_ml/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:37:34] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1745056857893/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/users/ysu13/miniforge3/envs/drug_sensitivity_ml/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/users/ysu13/miniforge3/envs/drug_sen